In [1]:
import os
import pickle

import numpy as np
import pandas as pd

from sklearn.utils import Bunch

from efaar_benchmarking.constants import COMPOUND_CONCENTRATIONS
from efaar_benchmarking.efaar import pca_centerscale_on_controls
from efaar_benchmarking.benchmarking import known_relationship_benchmark
from efaar_benchmarking.benchmarking import compound_gene_benchmark, BenchmarkConfig

In [2]:
# be sure to download these files and place them in the data directory
rxrx3_metadata = pd.read_csv("data/metadata_rxrx3_core.csv")  # visit https://rxrx3.rxrx.ai/downloads
openphenom_embeddings = pd.read_parquet("data/OpenPhenom_rxrx3-core_embeddings.parquet")  # download from huggingface

/var/folders/rh/2pzrb9394871v5556j113wwh0000gr/T/ipykernel_12082/1030283650.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  rxrx3_metadata = pd.read_csv("data/metadata_rxrx3_core.csv")  # visit https://rxrx3.rxrx.ai/downloads


In [3]:
rxrx3_metadata["perturbation"] = rxrx3_metadata["treatment"].apply(lambda x: x.split("_")[0] if "_control" not in x else x)

In [4]:
display(openphenom_embeddings)
display(rxrx3_metadata)

,well_id,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_374,feature_375,feature_376,feature_377,feature_378,feature_379,feature_380,feature_381,feature_382,feature_383
0,compound-001_1_AA15,0.043538,0.127778,-0.024083,-0.089810,-0.062772,0.303842,-0.307655,-0.157283,-0.280813,...,0.019589,-0.122266,0.058018,-0.120580,-0.139133,-0.044054,0.005494,-0.089248,0.215839,-0.153621
1,compound-001_1_AA16,0.044586,0.126481,-0.082607,-0.058849,-0.071924,0.316631,-0.312535,-0.107345,-0.430733,...,0.022487,-0.082299,0.053328,-0.025058,-0.140843,-0.053769,0.013818,-0.039017,0.191350,-0.155693
2,compound-001_1_AA18,0.036950,0.134374,0.023871,-0.100079,-0.046217,0.298770,-0.285384,-0.163665,-0.125366,...,0.047107,-0.145937,0.065314,-0.106710,-0.160478,-0.027014,0.026687,-0.108856,0.217319,-0.149561
3,compound-001_1_AA20,0.035557,0.137274,-0.022724,-0.096918,-0.073478,0.319549,-0.306399,-0.137685,-0.333849,...,0.047692,-0.104900,0.051111,-0.065427,-0.153742,-0.030151,-0.003287,-0.062774,0.214928,-0.168508
4,compound-001_1_AA25,0.041030,0.139127,-0.044030,-0.100211,-0.083253,0.296135,-0.305459,-0.161602,-0.331096,...,0.029277,-0.124027,0.067315,-0.078809,-0.144767,-0.041890,0.019115,-0.057309,0.189642,-0.175702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222596,gene-176_9_Z08,0.054115,0.142036,0.062627,-0.151489,-0.067934,0.382506,-0.279984,0.022767,0.325684,...,-0.063034,-0.159469,0.060406,-0.172644,-0.097095,-0.039370,0.130357,-0.076385,0.180803,-0.112373
222597,gene-176_9_Z23,0.036270,0.133238,-0.059697,-0.072660,-0.065886,0.345482,-0.314586,-0.154534,-0.406769,...,0.014219,-0.167755,0.056927,-0.124537,-0.168807,-0.018708,0.021021,-0.070389,0.190481,-0.187019
222598,gene-176_9_Z26,0.034006,0.144114,-0.069128,-0.066137,-0.046017,0.289187,-0.330449,-0.167187,-0.370520,...,-0.011062,-0.136820,0.066358,-0.098233,-0.197760,-0.031606,0.031998,-0.075168,0.140559,-0.166364
222599,gene-176_9_Z43,0.043700,0.120568,-0.012223,-0.066521,-0.079170,0.317472,-0.307256,-0.139971,-0.246299,...,0.036988,-0.145052,0.045938,-0.106476,-0.160615,0.016202,0.037935,-0.089957,0.196392,-0.189786


,well_id,experiment_name,plate,address,gene,treatment,SMILES,concentration,perturbation_type,cell_type,perturbation
0,compound-003_11_AD37,compound-003,11,AD37,NaN,Phloretin,"OC1=CC=C(CCC(=O)C2=C(O)C=C(O)C=C2O)C=C1 |c:9,1...",0.025,COMPOUND,HUVEC,Phloretin
1,compound-003_35_Y15,compound-003,35,Y15,NaN,Clozapine,CN1CCN(CC1)C1=NC2=C(NC3=C1C=CC=C3)C=CC(Cl)=C2 ...,2.500,COMPOUND,HUVEC,Clozapine
2,compound-001_19_D20,compound-001,19,D20,NaN,Dequalinium,CC1=[N+](CCCCCCCCCC[N+]2=C(C)C=C(N)C3=CC=CC=C2...,0.250,COMPOUND,HUVEC,Dequalinium
3,compound-001_11_U08,compound-001,11,U08,NaN,EMPTY_control,NaN,NaN,COMPOUND,HUVEC,EMPTY_control
4,compound-004_43_B08,compound-004,43,B08,NaN,CRISPR_control,NaN,NaN,COMPOUND,HUVEC,CRISPR_control
...,...,...,...,...,...,...,...,...,...,...,...
222596,gene-176_9_B09,gene-176,9,B09,EMPTY_control,EMPTY_control,NaN,NaN,CRISPR,HUVEC,EMPTY_control
222597,gene-176_6_T13,gene-176,6,T13,PEX6,PEX6_guide_2,NaN,NaN,CRISPR,HUVEC,PEX6
222598,gene-176_8_I27,gene-176,8,I27,ACAA2,ACAA2_guide_4,NaN,NaN,CRISPR,HUVEC,ACAA2
222599,gene-176_4_C35,gene-176,4,C35,FBL,FBL_guide_5,NaN,NaN,CRISPR,HUVEC,FBL


In [5]:
results_output_file = "data/RESULT_openphenom__pcacs.pickle"

In [6]:
embeddings_mrged = rxrx3_metadata.merge(
    openphenom_embeddings.rename(columns={"well_id": "external_well_id"}).groupby("external_well_id").mean(), 
    left_on="well_id", 
    right_index=True,
)

In [7]:
feature_columns = [c for c in embeddings_mrged.columns if c.startswith("feature_")]
metadata_columns = [c for c in embeddings_mrged.columns if not c.startswith("feature_")]

pert_colname = "perturbation"
experiment_colname = "experiment_name"
control_key = "EMPTY_control"

In [8]:
print("fitting aligner...")
X = embeddings_mrged[feature_columns].astype(float).values
embeddings_pcacs = pca_centerscale_on_controls(
    X, embeddings_mrged[metadata_columns], pert_col=pert_colname, batch_col=experiment_colname, control_key=control_key
)

assert embeddings_mrged[metadata_columns].shape[0] == embeddings_pcacs.shape[0]

new_metadata = embeddings_mrged[metadata_columns].copy().reset_index()
new_features = pd.DataFrame(embeddings_pcacs, columns=[f"feature_{i}" for i in range(embeddings_pcacs.shape[1])])
aligned_embeddings = pd.concat([new_metadata, new_features], axis=1)

fitting aligner...


In [9]:
assert aligned_embeddings.feature_0.isna().sum() == 0

# remove controls from henceforth analysis
merged = aligned_embeddings[
    ~(
        (aligned_embeddings["perturbation_type"] == "COMPOUND")
        & (aligned_embeddings[pert_colname].str.contains("_control"))
    )
]

# aggregate to perturbation-level
agg_func = {col: "mean" for col in merged.columns if col.startswith("feature_")}
map_data = (
    merged.groupby(["perturbation_type", pert_colname, "concentration"], dropna=False)
    .agg(agg_func)
    .reset_index()
)
map_data = map_data[map_data.concentration.isin(COMPOUND_CONCENTRATIONS) | map_data.concentration.isna()]
features_cols = [col for col in map_data.columns if col.startswith("feature_")]
metadata_cols = [col for col in map_data.columns if col not in features_cols]

assert map_data.feature_0.isna().sum() == 0

In [10]:
pert_signal_pval_cutoff = 0.05
recall_thr_pairs = [(0.05, 0.95)]

print("Computing recall...")
bmdb_metrics = known_relationship_benchmark(
    Bunch(metadata=map_data[metadata_cols], features=map_data[features_cols]),
    recall_thr_pairs=recall_thr_pairs,
    pert_col=pert_colname,
    log_stats=True,
)
print("Recall Results", bmdb_metrics[list(bmdb_metrics.columns)[::-1]])

Computing recall...
14090 perturbations exist in the map.
1209 relationships are used from the benchmark source CORUM
958 relationships are used from the benchmark source HuMAP
569 relationships are used from the benchmark source Reactome
495 relationships are used from the benchmark source SIGNOR
1737 relationships are used from the benchmark source StringDB
Recall Results      source  recall_0.05_0.95  query_distribution_size  null_distribution_size
0     CORUM          0.648470                     1209                99257005
1     HuMAP          0.723382                      958                99257005
2  Reactome          0.418278                      569                99257005
3    SIGNOR          0.385859                      495                99257005
4  StringDB          0.579159                     1737                99257005


In [11]:
all_compound_results = []
for seed in range(10):
    compound_results = compound_gene_benchmark(
        Bunch(metadata=map_data[metadata_cols], features=map_data[features_cols]),
        check_random=False,
        config=BenchmarkConfig(random_seed=seed),
    )
    compound_results["seed"] = seed
    all_compound_results.append(compound_results)
compound_results = pd.concat(all_compound_results)

In [12]:
mean_results = compound_results.groupby('concentration').agg(
    avg_precision_mean=('average_precision', 'mean'),
    avg_precision_std=('average_precision', 'std'),
    baseline_mean=('average_precision_baseline', 'mean'),
    baseline_std=('average_precision_baseline', 'std')
).reset_index()
mean_results["concentration"] = mean_results["concentration"].astype(str)

mean_results_auc = compound_results.groupby('concentration').agg(
    auc_roc_mean=('auc_roc', 'mean'),
    auc_roc_std=('auc_roc', 'std'),
    baseline_auc_mean=('auc_roc_baseline', 'mean'),
    baseline_auc_std=('auc_roc_baseline', 'std')
).reset_index()
mean_results_auc["concentration"] = mean_results_auc["concentration"].astype(str)

In [13]:
import plotly.express as px
import plotly.graph_objects as go

# Define the metrics and their corresponding labels
metrics = [
    {
        "mean_results": mean_results,
        "y_mean": "avg_precision_mean",
        "y_std": "avg_precision_std",
        "baseline_mean": "baseline_mean",
        "baseline_std": "baseline_std",
        "title": "Average Precision vs Baseline Precision with Error Bars",
        "yaxis_title": "Avg. Precision"
    },
    {
        "mean_results": mean_results_auc,
        "y_mean": "auc_roc_mean",
        "y_std": "auc_roc_std",
        "baseline_mean": "baseline_auc_mean",
        "baseline_std": "baseline_auc_std",
        "title": "AUC ROC vs Baseline AUC ROC with Error Bars",
        "yaxis_title": "AUC ROC"
    }
]

# Loop through the metrics and create the figures
for metric in metrics:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=metric["mean_results"]['concentration'],
        y=metric["mean_results"][metric["y_mean"]],
        error_y=dict(type='data', array=metric["mean_results"][metric["y_std"]]),
        mode='lines+markers',
        name="OpenPhenom",
    ))

    fig.add_trace(go.Scatter(
        x=metric["mean_results"]['concentration'],
        y=metric["mean_results"][metric["baseline_mean"]],
        error_y=dict(type='data', array=metric["mean_results"][metric["baseline_std"]]),
        mode='lines+markers',
        name="Random Baseline",
    ))

    fig.update_layout(
        title=metric["title"],
        xaxis_title='Concentration',
        yaxis_title=metric["yaxis_title"],
        template='plotly_dark'
    )

    fig.show()